# MVP – Sistema de Suporte à Decisão para Implantação de Nova Unidade de Dados

## Contexto

Empresas responsáveis por armazenamento e processamento de dados em nuvem precisam avaliar diversos fatores antes de implantar uma nova unidade de infraestrutura, como custos, prazo de implantação, disponibilidade de recursos e riscos do projeto.

## Objetivo

Desenvolver um Mínimo Produto Viável (MVP) de um Sistema de Suporte à Decisão capaz de analisar características de projetos de infraestrutura e apoiar a avaliação da viabilidade de implantação de uma nova unidade de armazenamento e processamento de dados.

O sistema considerará principalmente as dimensões de **custo, tempo e risco**, utilizando dados históricos de projetos de infraestrutura disponibilizados no Kaggle.

In [1]:
!pip install -q kagglehub

In [2]:
import pandas as pd
import os
import glob
import kagglehub

# Download da base do Kaggle
caminho = kagglehub.dataset_download(
    "ziya07/engineering-project-cost-feasibility-dataset"
)

# Localiza o arquivo CSV
arquivos_csv = glob.glob(
    os.path.join(caminho, "**", "*.csv"),
    recursive=True
)

print("Arquivos encontrados:")
print(arquivos_csv)

Using Colab cache for faster access to the 'engineering-project-cost-feasibility-dataset' dataset.
Arquivos encontrados:
['/kaggle/input/engineering-project-cost-feasibility-dataset/Engineering_Cost_Feasibility_Dataset.csv']


In [3]:
df = pd.read_csv(arquivos_csv[0])

print("Dimensão da base:", df.shape)
display(df.head())

Dimensão da base: (3245, 13)


,Project_ID,Project_Name,Project_Type,Scope_Complexity,Estimated_Cost_USD,Resource_Allocation_Score,Risk_Assessment_Score,Environmental_Impact_Score,Historical_Cost_Deviation_%,Stakeholder_Priority_Score,Time_Estimate_Days,Feasibility_Label,Scope_Complexity_Numeric
0,P0001,Urban Flyover,Road,Medium,2692743,57,68,69,0.00,0.50,268,Feasible,2
1,P0002,Urban Flyover,Road,Medium,5304572,51,38,22,1.69,0.68,375,Feasible,2
2,P0003,Industrial Complex,Road,High,5926484,90,50,62,0.30,0.62,219,Not Feasible,3
3,P0004,Smart Solar Grid,Road,Medium,3234489,95,58,76,3.28,0.53,174,Borderline,2
4,P0005,Smart Solar Grid,Building,High,2570006,71,62,69,9.08,0.59,305,Not Feasible,3


In [4]:
print("=== COLUNAS DA BASE ===")

for coluna in df.columns:
    print(coluna)

print("\n=== VALORES AUSENTES ===")
print(df.isnull().sum())

=== COLUNAS DA BASE ===
Project_ID
Project_Name
Project_Type
Scope_Complexity
Estimated_Cost_USD
Resource_Allocation_Score
Risk_Assessment_Score
Environmental_Impact_Score
Historical_Cost_Deviation_%
Stakeholder_Priority_Score
Time_Estimate_Days
Feasibility_Label
Scope_Complexity_Numeric

=== VALORES AUSENTES ===
Project_ID                     0
Project_Name                   0
Project_Type                   0
Scope_Complexity               0
Estimated_Cost_USD             0
Resource_Allocation_Score      0
Risk_Assessment_Score          0
Environmental_Impact_Score     0
Historical_Cost_Deviation_%    0
Stakeholder_Priority_Score     0
Time_Estimate_Days             0
Feasibility_Label              0
Scope_Complexity_Numeric       0
dtype: int64


## Critérios de análise

O MVP será estruturado a partir das três dimensões centrais da decisão de implantação:

- **Custo:** valor estimado para implantação da nova unidade;
- **Risco:** nível de risco associado ao projeto;
- **Tempo:** prazo estimado para implantação.

Os dados históricos do Kaggle serão utilizados como referência para comparar o novo projeto com outros projetos de infraestrutura.

Como a base não é específica de data centers, ela será utilizada como referência geral de projetos de infraestrutura, sem assumir que os registros representam unidades de armazenamento em nuvem.

## Benchmark histórico

A base do Kaggle será utilizada para estabelecer referências de custo, risco e prazo dos projetos de infraestrutura.

Serão analisados os valores mínimo, máximo, média, mediana e quartis dessas três dimensões. Esses valores servirão como base para posicionar o projeto da nova unidade em relação ao histórico disponível.

In [5]:
# Variáveis centrais para análise da nova unidade
variaveis_principais = [
    "Estimated_Cost_USD",
    "Risk_Assessment_Score",
    "Time_Estimate_Days"
]

benchmark = df[variaveis_principais].describe(
    percentiles=[0.25, 0.50, 0.75]
).round(2)

benchmark

,Estimated_Cost_USD,Risk_Assessment_Score,Time_Estimate_Days
count,3245.00,3245.00,3245.00
mean,4014063.66,64.61,357.51
std,1708536.46,20.21,141.04
min,1000404.00,30.00,120.00
25%,2564718.00,47.00,232.00
50%,3994200.00,64.00,356.00
75%,5450812.00,82.00,480.00
max,6998223.00,99.00,599.00


In [6]:
def calcular_percentil(valor, serie):
    return (serie <= valor).mean() * 100

## MVP — Avaliação de uma Nova Unidade de Infraestrutura de Dados

O sistema recebe três informações do projeto:

- **Custo estimado de implantação**
- **Nível de risco**
- **Prazo estimado de implantação**

Cada valor é comparado com os 3.245 projetos históricos da base do Kaggle e convertido em um percentil. Quanto maior o percentil, maior a exposição relativa do novo projeto naquela dimensão.

Para este MVP, custo, risco e tempo possuem o mesmo peso no índice geral. O resultado será utilizado para classificar o cenário como **Favorável, Atenção ou Crítico** e gerar recomendações de planejamento para a implantação.

In [7]:
def avaliar_nova_unidade(custo_usd, risco, tempo_dias):

    # Posicionamento do projeto em relação à base histórica
    score_custo = calcular_percentil(
        custo_usd,
        df["Estimated_Cost_USD"]
    )

    score_risco = calcular_percentil(
        risco,
        df["Risk_Assessment_Score"]
    )

    score_tempo = calcular_percentil(
        tempo_dias,
        df["Time_Estimate_Days"]
    )

    # Índice geral com pesos iguais
    indice_geral = (
        score_custo +
        score_risco +
        score_tempo
    ) / 3

    print("=== SISTEMA DE SUPORTE À DECISÃO ===")
    print(f"Custo estimado: US$ {custo_usd:,.2f}")
    print(f"Prazo estimado: {tempo_dias:.0f} dias")
    print(f"Risco informado: {risco:.1f}/100")

    print("\n=== POSIÇÃO EM RELAÇÃO AOS PROJETOS HISTÓRICOS ===")
    print(f"Exposição de custo: {score_custo:.1f}%")
    print(f"Exposição de risco: {score_risco:.1f}%")
    print(f"Exposição de prazo: {score_tempo:.1f}%")
    print(f"Índice geral: {indice_geral:.1f}%")

    # Classificação geral
    if indice_geral < 33:
        situacao = "FAVORÁVEL"
    elif indice_geral < 67:
        situacao = "ATENÇÃO"
    else:
        situacao = "CRÍTICA"

    print(f"\nSituação do projeto: {situacao}")

    print("\n=== RECOMENDAÇÕES PARA IMPLANTAÇÃO ===")

    if score_custo >= 67:
        print("- Revisar orçamento, fornecedores e criar reserva de contingência.")
    else:
        print("- Custo dentro de uma faixa administrável frente aos projetos históricos.")

    if score_risco >= 67:
        print("- Elaborar plano formal de mitigação de riscos antes da implantação.")
    else:
        print("- Manter monitoramento periódico dos riscos do projeto.")

    if score_tempo >= 67:
        print("- Dividir a implantação em etapas e acompanhar marcos críticos do cronograma.")
    else:
        print("- Prazo está em faixa administrável frente aos projetos históricos.")

    if situacao == "FAVORÁVEL":
        print("- O projeto apresenta condições favoráveis para avançar ao planejamento detalhado.")
    elif situacao == "ATENÇÃO":
        print("- Recomenda-se revisar os principais pontos de atenção antes da aprovação definitiva.")
    else:
        print("- Recomenda-se não iniciar a implantação sem revisão do planejamento.")

## Teste do MVP

Para demonstrar o funcionamento do sistema, será avaliado um cenário hipotético de implantação de uma nova unidade de infraestrutura de dados.

O cenário considera:

- **Custo estimado:** US$ 5.500.000
- **Risco:** 80/100
- **Prazo estimado:** 480 dias

In [8]:
avaliar_nova_unidade(
    custo_usd=5500000,
    risco=80,
    tempo_dias=480
)

=== SISTEMA DE SUPORTE À DECISÃO ===
Custo estimado: US$ 5,500,000.00
Prazo estimado: 480 dias
Risco informado: 80.0/100

=== POSIÇÃO EM RELAÇÃO AOS PROJETOS HISTÓRICOS ===
Exposição de custo: 75.7%
Exposição de risco: 72.3%
Exposição de prazo: 75.1%
Índice geral: 74.4%

Situação do projeto: CRÍTICA

=== RECOMENDAÇÕES PARA IMPLANTAÇÃO ===
- Revisar orçamento, fornecedores e criar reserva de contingência.
- Elaborar plano formal de mitigação de riscos antes da implantação.
- Dividir a implantação em etapas e acompanhar marcos críticos do cronograma.
- Recomenda-se não iniciar a implantação sem revisão do planejamento.


## Conclusão

O MVP desenvolvido utiliza dados históricos de 3.245 projetos de infraestrutura do Kaggle para apoiar o planejamento da implantação de uma nova unidade de armazenamento e processamento de dados.

O sistema compara **custo, risco e tempo** do novo projeto com a distribuição histórica da base e calcula um índice geral de exposição. A partir desse resultado, o projeto é classificado como **Favorável, Atenção ou Crítico**, acompanhado de recomendações para o planejamento da implantação.

No cenário demonstrado, com custo estimado de US$ 5,5 milhões, risco de 80/100 e prazo de 480 dias, o índice geral foi de 74,4%, classificando o projeto como **Crítico**.

Nesse cenário, o sistema recomenda revisar orçamento e fornecedores, estabelecer reserva de contingência, elaborar um plano de mitigação de riscos e dividir a implantação em etapas com acompanhamento dos marcos críticos do cronograma.

O MVP funciona como uma ferramenta de apoio à decisão e não substitui a avaliação dos responsáveis pelo projeto. Os pesos iguais entre custo, risco e tempo e os limites de classificação adotados são premissas definidas para esta versão inicial e podem ser ajustados conforme as prioridades da empresa.